# Import Required Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

# Create Folder for Saving Outputs

In [3]:
os.makedirs("generated_images", exist_ok=True)
os.makedirs("interpolations", exist_ok=True)

# Data Preparation

In [ ]:
batch_size = 128
image_size = 64

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.CIFAR10(
    root="./data",
    download=True,
    transform=transform
)

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

# Latent Space Definition

In [5]:
latent_dim = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Generator Network(DCGAN)

In [24]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

# Discriminator Network

In [25]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*8*8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize Generator and Discriminator

In [30]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0)

generator.apply(weights_init)
discriminator.apply(weights_init)

# Define Loss Function and Optimizers

In [32]:
criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Train DCGAN

In [ ]:
noise = torch.randn(4, latent_dim,1,1).to(device)

fake = generator(noise)

print(fake.shape)
print(discriminator(fake).shape)

In [ ]:
epochs = 20

for epoch in range(epochs):

    for real_images, _ in dataloader:

        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        real_labels = torch.full((batch_size,1),0.9).to(device)
        fake_labels = torch.zeros(batch_size,1).to(device)
        optimizer_D.zero_grad()

        outputs = discriminator(real_images)
        loss_real = criterion(outputs, real_labels)

        noise = torch.randn(batch_size, latent_dim,1,1).to(device)
        fake_images = generator(noise)

        outputs = discriminator(real_images)
        loss_fake = criterion(outputs, fake_labels)

        d_loss = loss_real + loss_fake
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()

        outputs = discriminator(fake_images)
        g_loss = criterion(outputs, real_labels)

        g_loss.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{epochs}] D Loss: {d_loss.item()} G Loss: {g_loss.item()}")

    if (epoch+1) % 5 == 0:

        noise = torch.randn(16, latent_dim,1,1).to(device)
        fake_images = generator(noise)

        save_image(
            fake_images,
            f"generated_images/epoch_{epoch+1}.png",
            normalize=True
        )

# Generate Artistic Samples

In [ ]:
noise = torch.randn(10, latent_dim,1,1).to(device)

generated_images = generator(noise)

save_image(generated_images, "generated_images/artistic_samples.png", normalize=True)

grid = make_grid(generated_images.cpu(), nrow=5, normalize=True)

plt.figure(figsize=(8,4))
plt.imshow(np.transpose(grid,(1,2,0)))
plt.axis("off")
plt.show()

# Latent Space Interpolation

In [ ]:
z1 = torch.randn(1, latent_dim,1,1).to(device)
z2 = torch.randn(1, latent_dim,1,1).to(device)

alphas = torch.linspace(0,1,10)

images = []

for alpha in alphas:
    z = (1-alpha)*z1 + alpha*z2
    img = generator(z)
    images.append(img)

images = torch.cat(images)

save_image(images, "interpolations/latent_interpolation.png", normalize=True, nrow=10)

grid = make_grid(images.cpu(), nrow=10, normalize=True)

plt.figure(figsize=(15,3))
plt.imshow(np.transpose(grid,(1,2,0)))
plt.axis("off")
plt.show()

# Latent Space Visualization

In [ ]:
from sklearn.decomposition import PCA

num_samples = 200

latent_vectors = torch.randn(num_samples, latent_dim)

latent_np = latent_vectors.numpy()

pca = PCA(n_components=2)
latent_2d = pca.fit_transform(latent_np)

plt.figure(figsize=(6,6))
plt.scatter(latent_2d[:,0], latent_2d[:,1], alpha=0.7)
plt.title("Latent Space Visualization (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.savefig("latent_space_pca.png")
plt.show()

# Observations

1. GAN successfully generates new images from random noise vectors.
2. Different latent vectors produce diverse artistic outputs.
3. Interpolating between latent vectors produces smooth transitions.
4. Generated images are not copies of training images but new variations.